# Experiment 11: Multi-Approach Sentiment Analysis System for Reviews and Social Media

### Problem Statement:
Create a sentiment analysis application that classifies customer reviews or social media posts into positive, negative, or neutral sentiments using NLP and machine learning techniques.


In [1]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from transformers import pipeline

# Ensure VADER lexicon is downloaded
nltk.download('vader_lexicon', quiet=True)

print("Sentiment analysis engines and dependencies ready.")


/Users/jeet/miniconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentiment analysis engines and dependencies ready.


In [2]:
review_data = [
    # Positive Reviews
    ("The battery life on this laptop is absolutely incredible, easily lasting two full days!", "Positive"),
    ("Exceptional customer service and lightning fast delivery. Highly recommended!", "Positive"),
    ("The picture quality of this OLED display is breathtaking with deep rich blacks.", "Positive"),
    ("An absolute masterpiece of audio engineering; crisp highs and deep punchy bass.", "Positive"),
    ("I love the sleek modern design and intuitive user interface of this software.", "Positive"),
    ("The restaurant served delightfully fresh seafood and the dessert was divine.", "Positive"),

    # Negative Reviews
    ("Terrible build quality; the hinge snapped after only three days of gentle use.", "Negative"),
    ("The smartphone constantly overheats and shuts down randomly during calls. Avoid!", "Negative"),
    ("Extremely disappointed with the awful software bugs and non-responsive customer support.", "Negative"),
    ("The hotel room smelled moldy, the air conditioner was broken, and staff was rude.", "Negative"),
    ("Complete waste of money; the device failed to turn on right out of the box.", "Negative"),
    ("The food was cold, horribly greasy, and arrived two hours after ordering.", "Negative"),

    # Neutral Reviews
    ("The package arrived on Tuesday as scheduled in a standard cardboard box.", "Neutral"),
    ("The device dimensions are 15 inches wide and weighs approximately two kilograms.", "Neutral"),
    ("The television has three HDMI ports, one optical port, and includes a black remote.", "Neutral"),
    ("The operating system updated automatically at midnight without user confirmation.", "Neutral"),
    ("The restaurant offers both indoor and outdoor patio seating options on weekends.", "Neutral"),
    ("The application requires four gigabytes of available storage for installation.", "Neutral")
]

df_reviews = pd.DataFrame(review_data, columns=["Review", "True_Sentiment"])
print(f"Total reviews: {len(df_reviews)}")
print("\n--- Class Counts ---")
print(df_reviews["True_Sentiment"].value_counts())


Total reviews: 18

--- Class Counts ---
True_Sentiment
Positive    6
Negative    6
Neutral     6
Name: count, dtype: int64


In [3]:
vader = SentimentIntensityAnalyzer()

def classify_vader(text):
    scores = vader.polarity_scores(text)
    comp = scores['compound']
    if comp >= 0.05:
        label = "Positive"
    elif comp <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return label, comp

vader_preds = []
vader_compounds = []
textblob_polarities = []

for text in df_reviews["Review"]:
    lbl, comp = classify_vader(text)
    blob = TextBlob(text)
    vader_preds.append(lbl)
    vader_compounds.append(comp)
    textblob_polarities.append(blob.sentiment.polarity)

df_reviews["VADER_Pred"] = vader_preds
df_reviews["VADER_Compound"] = vader_compounds
df_reviews["TextBlob_Polarity"] = textblob_polarities

acc_vader = accuracy_score(df_reviews["True_Sentiment"], df_reviews["VADER_Pred"])
print(f"VADER Lexicon-Based Accuracy: {acc_vader*100:.2f}%")
print("\n--- Sample VADER Predictions ---")
print(df_reviews[["Review", "True_Sentiment", "VADER_Pred", "VADER_Compound"]].head(6).to_string())


VADER Lexicon-Based Accuracy: 100.00%

--- Sample VADER Predictions ---
                                                                                    Review True_Sentiment VADER_Pred  VADER_Compound
0  The battery life on this laptop is absolutely incredible, easily lasting two full days!       Positive   Positive          0.4534
1            Exceptional customer service and lightning fast delivery. Highly recommended!       Positive   Positive          0.3367
2          The picture quality of this OLED display is breathtaking with deep rich blacks.       Positive   Positive          0.7650
3          An absolute masterpiece of audio engineering; crisp highs and deep punchy bass.       Positive   Positive          0.6249
4            I love the sleek modern design and intuitive user interface of this software.       Positive   Positive          0.6369
5             The restaurant served delightfully fresh seafood and the dessert was divine.       Positive   Positive          0.86

In [4]:
tfidf_pipe = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', min_df=1)
X_tfidf = tfidf_pipe.fit_transform(df_reviews["Review"])
y_labels = df_reviews["True_Sentiment"]

ml_clf = LogisticRegression(C=1.0, random_state=42)
ml_clf.fit(X_tfidf, y_labels)

df_reviews["ML_Pred"] = ml_clf.predict(X_tfidf)
acc_ml = accuracy_score(df_reviews["True_Sentiment"], df_reviews["ML_Pred"])
print(f"Supervised ML (TF-IDF + LogReg) Accuracy: {acc_ml*100:.2f}%")


Supervised ML (TF-IDF + LogReg) Accuracy: 100.00%


In [5]:
hf_classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

hf_preds = []
hf_scores = []

for text in df_reviews["Review"]:
    res = hf_classifier(text)[0]
    # DistilBERT SST-2 is binary (POSITIVE / NEGATIVE); map score thresholds
    label = "Positive" if res["label"] == "POSITIVE" else "Negative"
    # If confidence is low or text has no affective tone, can represent neutral
    hf_preds.append(label)
    hf_scores.append(round(res["score"], 4))

df_reviews["HF_Pred"] = hf_preds
df_reviews["HF_Confidence"] = hf_scores

print("--- Hugging Face DistilBERT Predictions (Excerpt) ---")
print(df_reviews[["Review", "True_Sentiment", "HF_Pred", "HF_Confidence"]].head(6).to_string())


Loading weights:   0%|                                 | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 104/104 [00:00<00:00, 8728.52it/s]

--- Hugging Face DistilBERT Predictions (Excerpt) ---
                                                                                    Review True_Sentiment   HF_Pred  HF_Confidence
0  The battery life on this laptop is absolutely incredible, easily lasting two full days!       Positive  Positive         0.9995
1            Exceptional customer service and lightning fast delivery. Highly recommended!       Positive  Positive         0.9999
2          The picture quality of this OLED display is breathtaking with deep rich blacks.       Positive  Positive         0.9997
3          An absolute masterpiece of audio engineering; crisp highs and deep punchy bass.       Positive  Positive         0.9998
4            I love the sleek modern design and intuitive user interface of this software.       Positive  Positive         0.9999
5             The restaurant served delightfully fresh seafood and the dessert was divine.       Positive  Positive         0.9999


In [6]:
edge_cases = [
    "The camera is great, but the battery died immediately and ruined my trip.",
    "This budget laptop is definitely not bad at all for the price.",
    "I had very high expectations, but the sequel completely failed to impress.",
    "The new software update didn't disappoint in the slightest."
]

edge_records = []
for text in edge_cases:
    v_lbl, v_comp = classify_vader(text)
    tb_pol = TextBlob(text).sentiment.polarity
    h_res = hf_classifier(text)[0]
    
    edge_records.append({
        "Complex Linguistic Input": text,
        "VADER Label": v_lbl,
        "VADER Score": round(v_comp, 3),
        "TextBlob Polarity": round(tb_pol, 3),
        "DistilBERT Label": h_res["label"],
        "DistilBERT Confidence": round(h_res["score"], 3)
    })

df_edge = pd.DataFrame(edge_records)
print("--- Diagnostic Error Analysis on Complex Sentences ---")
for _, r in df_edge.iterrows():
    print(f"Input : '{r['Complex Linguistic Input']}'")
    print(f"  VADER      : {r['VADER Label']} (comp: {r['VADER Score']})")
    print(f"  TextBlob   : Polarity {r['TextBlob Polarity']}")
    print(f"  DistilBERT : {r['DistilBERT Label']} (conf: {r['DistilBERT Confidence']})\n")


--- Diagnostic Error Analysis on Complex Sentences ---
Input : 'The camera is great, but the battery died immediately and ruined my trip.'
  VADER      : Negative (comp: -0.818)
  TextBlob   : Polarity 0.8
  DistilBERT : NEGATIVE (conf: 1.0)

Input : 'This budget laptop is definitely not bad at all for the price.'
  VADER      : Positive (comp: 0.676)
  TextBlob   : Polarity 0.35
  DistilBERT : POSITIVE (conf: 0.996)

Input : 'I had very high expectations, but the sequel completely failed to impress.'
  VADER      : Negative (comp: -0.164)
  TextBlob   : Polarity -0.146
  DistilBERT : NEGATIVE (conf: 1.0)

Input : 'The new software update didn't disappoint in the slightest.'
  VADER      : Positive (comp: 0.309)
  TextBlob   : Polarity 0.136
  DistilBERT : POSITIVE (conf: 0.998)



In [7]:
def analyze_sentiment_composite(text):
    # 1. VADER
    v_scores = vader.polarity_scores(text)
    v_label, v_comp = classify_vader(text)
    
    # 2. TextBlob
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    # 3. Transformer
    hf_res = hf_classifier(text)[0]
    
    print(f"===========================================================")
    print(f"REVIEW TEXT: '{text}'")
    print(f"===========================================================")
    print(f"1. VADER Engine:")
    print(f"   Classification : {v_label.upper()}")
    print(f"   Compound Score : {v_comp:+.4f} (Pos: {v_scores['pos']}, Neu: {v_scores['neu']}, Neg: {v_scores['neg']})")
    print(f"2. TextBlob Engine:")
    print(f"   Polarity Score : {polarity:+.4f} [-1.0 = Highly Neg, +1.0 = Highly Pos]")
    print(f"   Subjectivity   : {subjectivity:.4f} [0.0 = Objective, 1.0 = Subjective]")
    print(f"3. Neural Transformer (DistilBERT):")
    print(f"   Predicted Class: {hf_res['label']}")
    print(f"   Confidence     : {hf_res['score']*100:.2f}%")
    print(f"===========================================================\n")

# Test interactive composite function
analyze_sentiment_composite("This restaurant has outstanding pasta, although the waiting line was exhausting.")
analyze_sentiment_composite("The customer support team resolved my warranty claim in ten minutes!")


REVIEW TEXT: 'This restaurant has outstanding pasta, although the waiting line was exhausting.'
1. VADER Engine:
   Classification : POSITIVE
   Compound Score : +0.3612 (Pos: 0.258, Neu: 0.581, Neg: 0.161)
2. TextBlob Engine:
   Polarity Score : +0.0500 [-1.0 = Highly Neg, +1.0 = Highly Pos]
   Subjectivity   : 0.6875 [0.0 = Objective, 1.0 = Subjective]
3. Neural Transformer (DistilBERT):
   Predicted Class: POSITIVE
   Confidence     : 99.87%

REVIEW TEXT: 'The customer support team resolved my warranty claim in ten minutes!'
1. VADER Engine:
   Classification : POSITIVE
   Compound Score : +0.5707 (Pos: 0.343, Neu: 0.657, Neg: 0.0)
2. TextBlob Engine:
   Polarity Score : +0.0000 [-1.0 = Highly Neg, +1.0 = Highly Pos]
   Subjectivity   : 0.0000 [0.0 = Objective, 1.0 = Subjective]
3. Neural Transformer (DistilBERT):
   Predicted Class: NEGATIVE
   Confidence     : 86.90%

